# Детекция ботов по событиям внутри суточного окна

Задача: для каждой `cookie_id` предсказать `score` от 0 до 1 — вероятность, что кука принадлежит
сервису автоматизированного сбора данных. Метрика: Precision при Recall >= 0.70.

Результат: 0.744 (OOF) против 0.102 у baseline из `quickstart.ipynb`.

Что оказалось важным по ходу работы:

1. `captcha_shown` встречается только после конца окна наблюдения. Признак очень сильный, но
   использовать его нельзя — разбор в разделе 3.
2. Дубликаты событий не удаляю: среди кук с дублями ботов вдвое больше базового уровня.
3. Сильнее всего разделяют тайминг и поведение курсора.
4. Метрика шумная, поэтому считаю её на полном OOF-векторе, а усреднение по фолдам не использую.

## 1. Настройка и загрузка данных

In [1]:
import os, sys, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

sys.path.insert(0, '.')
from metric import precision_at_recall, recall_at_fpr   # официальная метрика организаторов
from src.features import build_features, clip_to_window, normalize_platform

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

DATE_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=DATE_COLS)
test = pd.read_csv('data/test.csv', parse_dates=DATE_COLS)
events = pd.read_csv('data/events.csv', parse_dates=['event_ts'])

print('train  ', train.shape)
print('test   ', test.shape)
print('events ', events.shape)
print('доля ботов в train:', round(train.target.mean(), 4))

train   (11091, 5)
test    (4909, 4)
events  (328905, 14)
доля ботов в train: 0.0811


## 2. Структура данных и временные окна

Смотрю, как соотносятся окна train и test, нет ли пересечения кук.

In [2]:
print('окно наблюдения всегда ровно сутки:',
      (train.window_end_ts - train.window_start_ts).nunique() == 1,
      (test.window_end_ts - test.window_start_ts).nunique() == 1)

print('\ntrain: дни', train.window_start_ts.dt.date.min(), '->', train.window_start_ts.dt.date.max())
print('test : дни', test.window_start_ts.dt.date.min(), '->', test.window_start_ts.dt.date.max())
print('\nпересечение cookie_id между train и test:', len(set(train.cookie_id) & set(test.cookie_id)))
print('дубликаты cookie_id: train =', train.cookie_id.duplicated().sum(),
      ', test =', test.cookie_id.duplicated().sum())

# доля ботов по дням — проверяем, нет ли тренда, из-за которого валидация поехала бы
by_day = train.groupby(train.window_start_ts.dt.date).target.agg(['size', 'mean']).round(4)
print('\nдоля ботов по дням train:')
print(by_day.to_string())

окно наблюдения всегда ровно сутки: True True

train: дни 2026-04-06 -> 2026-04-19
test : дни 2026-04-20 -> 2026-04-26

пересечение cookie_id между train и test: 0
дубликаты cookie_id: train = 0 , test = 0

доля ботов по дням train:
                 size    mean
window_start_ts              
2026-04-06        772  0.0881
2026-04-07        797  0.0740
2026-04-08        834  0.0839
2026-04-09        902  0.0732
2026-04-10        912  0.0899
2026-04-11        896  0.0882
2026-04-12        817  0.0820
2026-04-13        843  0.0629
2026-04-14        826  0.0847
2026-04-15        801  0.0811
2026-04-16        740  0.0811
2026-04-17        713  0.0926
2026-04-18        606  0.0858
2026-04-19        632  0.0665


Train занимает 14 суток (06–19 апреля), test — следующие 7 (20–26). Test целиком в будущем
относительно train, пересечения кук нет, доля ботов по дням стабильная.

Отсюда требования к валидации: нужно проверять устойчивость во времени (контроль на последних
днях train) и при этом иметь стабильную оценку (случайное разбиение). Обе схемы — в разделе 6.

## 3. Проверка на утечку из будущего

Задача требует: *«Признаки должны быть доступны на момент окончания окна наблюдения»*. Значит
допустимы только события с `window_start_ts <= event_ts < window_end_ts`. Проверяю, что лежит
за пределами окна.

In [3]:
meta_all = pd.concat([train.drop(columns='target'), test], ignore_index=True)
ev_all = events.merge(meta_all[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id', how='left')
ev_all['in_window'] = (ev_all.event_ts >= ev_all.window_start_ts) & (ev_all.event_ts < ev_all.window_end_ts)

print('событий всего      :', len(ev_all))
print('внутри окна        :', int(ev_all.in_window.sum()))
print('до начала окна     :', int((ev_all.event_ts < ev_all.window_start_ts).sum()))
print('после конца окна   :', int((ev_all.event_ts >= ev_all.window_end_ts).sum()))

split = ev_all.groupby(['eid', 'event_name']).in_window.agg(['size', 'sum'])
split['после_окна'] = split['size'] - split['sum']
split['доля_после_окна'] = (split['после_окна'] / split['size']).round(3)
print('\nраспределение событий по типам:')
print(split.rename(columns={'size': 'всего', 'sum': 'в_окне'}).to_string())

событий всего      : 328905
внутри окна        : 288126
до начала окна     : 0
после конца окна   : 40779

распределение событий по типам:
                           всего  в_окне  после_окна  доля_после_окна
eid event_name                                                       
100 search_results_view   100402   89798       10604            0.106
200 item_view             120817  108086       12731            0.105
210 photo_swipe            36517   33084        3433            0.094
220 seller_page_view       17403   15542        1861            0.107
300 contact_phone_show     11316   10288        1028            0.091
301 contact_chat_open       6125    5548         577            0.094
303 contact_message_sent    3081    2823         258            0.084
400 favorite_add           19049   17296        1753            0.092
500 login                   6267    5661         606            0.097
900 captcha_shown           7928       0        7928            1.000


`captcha_shown` целиком лежит за пределами окна — внутри окна таких событий ноль. Похоже, это
не случайность: показ капчи — реакция антибот-системы на поведение, которое мы и предсказываем.

Дальше смотрю, насколько сильный это признак.

In [4]:
# сколько раз капча была показана куке ПОСЛЕ конца окна (запрещённый признак)
post = ev_all[~ev_all.in_window]
cap = post[post.eid == 900].groupby('cookie_id').size().rename('n_captcha_after')
tgt = train.set_index('cookie_id').target

lab = tgt.to_frame().join(cap).fillna({'n_captcha_after': 0})
print('доля ботов среди куки С капчей после окна :',
      round(lab.loc[lab.n_captcha_after > 0, 'target'].mean(), 4),
      f'(n={int((lab.n_captcha_after > 0).sum())})')
print('доля ботов среди куки БЕЗ капчи           :',
      round(lab.loc[lab.n_captcha_after == 0, 'target'].mean(), 4))
print('базовая доля ботов                        :', round(train.target.mean(), 4))

доля ботов среди куки С капчей после окна : 0.771 (n=572)
доля ботов среди куки БЕЗ капчи           : 0.0435
базовая доля ботов                        : 0.0811


Среди кук, которым после окна показали капчу, ботов 77% против базовых 8%.

Признак не использую. Он недоступен в момент принятия решения, поэтому модель на нём в проде
работать не будет — там капчи ещё нет. Всё дальше считаю строго внутри окна, через
`clip_to_window` в `src/features.py`. Во что это обходится по метрике — в разделе 8.

## 4. Качество данных

Три вещи, которые нужно обработать осознанно: грязные категории, дубликаты и пропуски.

In [5]:
ev_win = clip_to_window(events, meta_all)   # дальше работаем только с событиями внутри окна

print('--- platform: одна и та же платформа записана по-разному ---')
print(ev_win.platform.value_counts().to_string())
print('\nпосле нормализации:')
print(normalize_platform(ev_win.platform).value_counts().to_string())

--- platform: одна и та же платформа записана по-разному ---
platform
desktop    40768
WEB        40330
Web        40269
web        39895
ANDROID    37597
Android    37357
android    37302
ios         3659
iphone      3656
IOS         3651
iOS         3642

после нормализации:


platform
web        120494
android    112256
desktop     40768
ios         14608


In [6]:
print('--- дубликаты событий ---')
dup_mask = ev_win.duplicated(subset=['cookie_id', 'event_ts', 'eid', 'item_id', 'search_query'], keep=False)
dup_cookies = ev_win.loc[dup_mask, 'cookie_id'].unique()
d = tgt.reindex(dup_cookies).dropna()
print('строк-дубликатов        :', int(dup_mask.sum()))
print('куки с дубликатами      :', len(dup_cookies))
print('доля ботов среди них    :', round(d.mean(), 4), f'(n={len(d)})')
print('базовая доля ботов      :', round(train.target.mean(), 4))

--- дубликаты событий ---


строк-дубликатов        : 8629
куки с дубликатами      : 3413
доля ботов среди них    : 0.1217 (n=2374)
базовая доля ботов      : 0.0811


Среди кук с дублирующимися событиями ботов около 16% против базовых 8%. Дубликаты несут сигнал,
поэтому я их не удаляю, а считаю признак `dup_event_share`.

In [7]:
print('--- пропуски: доля NaN по колонкам ---')
print(ev_win.isna().mean().round(3).to_string())

print('\n--- pointer_x заполнен не случайно: его просто нет на мобильных ---')
pl = normalize_platform(ev_win.platform)
print(ev_win.assign(pl=pl, miss=ev_win.pointer_x.isna()).groupby('pl').miss.mean().round(3).to_string())

--- пропуски: доля NaN по колонкам ---
cookie_id          0.000
event_ts           0.000
eid                0.000
event_name         0.000
platform           0.000
user_agent         0.000
item_id            0.331
item_category      0.078
item_location      0.050
seller_type        0.391
search_query       0.688
search_page        0.688
pointer_x          0.668
pointer_y          0.668
window_start_ts    0.000
window_end_ts      0.000

--- pointer_x заполнен не случайно: его просто нет на мобильных ---


pl
android    1.000
desktop    0.406
ios        1.000
web        0.407


`pointer_x/y` пусты всегда на android/ios и примерно в 41% случаев на web/desktop. То есть
«нет координат курсора» само по себе означает всего лишь «мобильное устройство».

Поэтому признаки курсора считаю только внутри web/desktop (блок F в `features.py`) — иначе они
вырождаются в индикатор платформы. В итоге это один из самых сильных признаков.

Остальные пропуски осмысленны: `search_query`/`search_page` есть только у событий выдачи,
`item_id` — только у событий с объявлением. Константой не заполняю, агрегаты считаю по нужному
подмножеству событий, а LightGBM работает с NaN сам.

## 5. Построение признаков

Признаки собраны в `src/features.py`, одна строка на куку.

| Группа | Идея | Примеры |
|---|---|---|
| A. Объём и интенсивность | сколько и как часто | `n_events`, `events_per_min` |
| B. Тайминг | человек ходит рывками, бот ровно | `dt_median`, `dt_p90`, `dt_cv`, `dt_long_share` |
| C. Типы событий | зачем пришли | `sh_contact`, `sh_engagement` |
| D. Разнообразие контента | что обходят | `loc_nunique`, `cat_entropy`, `top_cat_share` |
| E. UA и платформа | чем ходят | `ua_headless`, `platform_nunique` |
| F. Курсор (web/desktop) | двигают ли мышь | `pointer_coverage`, `pointer_x_std` |
| G. Суточный профиль | когда активны | `hour_entropy`, `sh_night` |
| H. Дубликаты | повторы как след автомата | `dup_event_share` |
| I. Метаданные куки | возраст куки | `cookie_age_days` |

Не строю: признаки по `captcha_shown` (раздел 3), «кука создана внутри окна» (таких нет,
получилась бы константа) и день недели (сигнала нет, а тест лежит в другой календарной неделе).

In [8]:
%time X_train = build_features(events, train)
%time X_test = build_features(events, test)

y = train.target.values
FEATURES = [c for c in X_train.columns if c != 'cookie_id']

# порядок строк обязан совпадать с meta — иначе метки разъедутся
assert (X_train.cookie_id.values == train.cookie_id.values).all()
assert (X_test.cookie_id.values == test.cookie_id.values).all()
assert list(X_train.columns) == list(X_test.columns)

print('\nпризнаков:', len(FEATURES), '| train:', X_train.shape, '| test:', X_test.shape)

CPU times: total: 7.39 s
Wall time: 7.62 s


CPU times: total: 3.28 s
Wall time: 3.37 s

признаков: 72 | train: (11091, 73) | test: (4909, 73)


In [9]:
# сравнение распределений train/test — заметный сдвиг означал бы, что валидация обманет
cmp = pd.DataFrame({
    'train_median': X_train[FEATURES].median(),
    'test_median': X_test[FEATURES].median(),
    'train_NaN': X_train[FEATURES].isna().mean().round(3),
    'test_NaN': X_test[FEATURES].isna().mean().round(3),
})
cmp['отн_сдвиг'] = ((cmp.test_median - cmp.train_median) / cmp.train_median.abs().replace(0, np.nan)).round(3)
print('наибольшие сдвиги медиан между train и test:')
print(cmp.reindex(cmp['отн_сдвиг'].abs().sort_values(ascending=False).index).head(10).to_string())

наибольшие сдвиги медиан между train и test:
                        train_median  test_median  train_NaN  test_NaN  отн_сдвиг
sh_seller_page_view         0.021739     0.027027      0.000     0.000      0.243
n_item_view                 5.000000     4.000000      0.000     0.000     -0.200
sh_favorite_add             0.034483     0.031250      0.000     0.000     -0.094
events_per_active_hour      4.750000     5.000000      0.000     0.000      0.053
search_page_mean            2.125000     2.222222      0.083     0.089      0.046
sh_engagement               0.200000     0.192308      0.000     0.000     -0.038
pages_per_query             0.777778     0.800000      0.083     0.089      0.029
dt_p90                   1400.350000  1362.000000      0.022     0.021     -0.027
dt_p10                     14.600000    14.200000      0.022     0.021     -0.027
sh_photo_swipe              0.102041     0.100000      0.000     0.000     -0.020


## 6. Схема валидации

Метрика определяется несколькими десятками объектов около порога и поэтому шумнее ROC-AUC.
Сначала я считал её на каждом фолде отдельно и усреднял — разброс между фолдами доходил до
±0.09, и на таком шуме полезный признак было не отличить от вредного.

Итоговая схема из двух частей:

1. **OOF по всей выборке** (основная). Стратифицированный 5-fold, предсказания собираю для всех
   объектов, метрику считаю один раз на всех 899 ботах, с повторами по нескольким сидам.
   Разброс падает до ±0.01.
2. **Time-holdout** (контроль). Обучение на первых 10 днях, проверка на последних 4. Ловит
   деградацию во времени, которую случайное разбиение не увидит.

Выводы делаю, когда обе схемы согласны.

In [10]:
DAYS = train.window_start_ts.dt.normalize()

def make_model(kind, seed, params=None):
    if kind == 'lgb':
        return lgb.LGBMClassifier(random_state=seed, n_jobs=-1, verbose=-1, **(params or {}))
    if kind == 'rf':
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=seed, n_jobs=-1)
    if kind == 'logreg':
        return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                             LogisticRegression(max_iter=2000, C=0.5, random_state=seed))
    raise ValueError(kind)

def prep(df, kind):
    # sklearn-модели не переваривают NaN, LightGBM обрабатывает их сам
    return df.fillna(df.median()) if kind == 'rf' else df

def oof_predict(X, cols, params=None, n_repeats=3, model='lgb', n_seeds=1):
    # Список OOF-векторов, по одному на повтор разбиения.
    outs = []
    for r in range(n_repeats):
        oof = np.zeros(len(y))
        for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=100 + r).split(X, y):
            p = np.zeros(len(va_idx))
            for s in range(n_seeds):
                mdl = make_model(model, seed=r * 10 + s, params=params)
                mdl.fit(prep(X.iloc[tr_idx][cols], model), y[tr_idx])
                p += mdl.predict_proba(prep(X.iloc[va_idx][cols], model))[:, 1]
            oof[va_idx] = p / n_seeds
        outs.append(oof)
    return outs

def evaluate(X, cols, params=None, n_repeats=3, model='lgb', n_seeds=1):
    # Метрика считается на ПОЛНОМ OOF-векторе (все 899 ботов), а не усреднением по фолдам.
    outs = oof_predict(X, cols, params, n_repeats, model, n_seeds)
    pr = [precision_at_recall(y, o) for o in outs]
    return dict(pr=np.mean(pr), pr_std=np.std(pr),
                roc=np.mean([roc_auc_score(y, o) for o in outs]),
                ap=np.mean([average_precision_score(y, o) for o in outs]),
                oof=outs[0])

def report(X, cols, name, **kw):
    r = evaluate(X, cols, **kw)
    print(f'{name:38s} P@R0.7 = {r["pr"]:.4f} +- {r["pr_std"]:.4f} | '
          f'ROC {r["roc"]:.4f} | PR-AUC {r["ap"]:.4f}')
    return r

def time_holdout(X, cols, params=None, model='lgb', n_val_days=4, n_seeds=3):
    u = np.sort(DAYS.unique())
    va = DAYS.isin(u[-n_val_days:]).values
    p = np.zeros(va.sum())
    for s in range(n_seeds):
        mdl = make_model(model, seed=s, params=params)
        mdl.fit(prep(X.loc[~va, cols], model), y[~va])
        p += mdl.predict_proba(prep(X.loc[va, cols], model))[:, 1]
    p /= n_seeds
    return precision_at_recall(y[va], p), roc_auc_score(y[va], p), int(y[va].sum())

u = np.sort(DAYS.unique())
print('time-holdout: обучение на', len(u) - 4, 'днях, валидация на', 4,
      f'днях ({DAYS.isin(u[-4:]).sum()} кук, {int(y[DAYS.isin(u[-4:]).values].sum())} ботов)')

time-holdout: обучение на 10 днях, валидация на 4 днях (2691 кук, 220 ботов)


## 7. Baseline и выбор модели

Baseline берётся ровно из `quickstart.ipynb` — RandomForest на двух признаках. Это точка отсчёта,
относительно которой видно вклад признаков и модели.

In [11]:
LGB_BASE = dict(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=30,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0)

print('базовая доля ботов (уровень случайного скоринга):', round(y.mean(), 4), '\n')
report(X_train, ['n_events', 'item_nunique'], 'baseline quickstart (2 признака, RF)', model='rf')
report(X_train, FEATURES, 'логрегрессия, все признаки', model='logreg')
report(X_train, FEATURES, 'LightGBM, все признаки', params=LGB_BASE)

базовая доля ботов (уровень случайного скоринга): 0.0811 



baseline quickstart (2 признака, RF)   P@R0.7 = 0.1021 +- 0.0013 | ROC 0.6280 | PR-AUC 0.2181


логрегрессия, все признаки             P@R0.7 = 0.5097 +- 0.0073 | ROC 0.9034 | PR-AUC 0.6516


LightGBM, все признаки                 P@R0.7 = 0.7367 +- 0.0096 | ROC 0.9273 | PR-AUC 0.7736


{'pr': np.float64(0.7366909573991959),
 'pr_std': np.float64(0.009553850631787758),
 'roc': np.float64(0.9272673238885698),
 'ap': np.float64(0.7736285518036695),
 'oof': array([0.02829334, 0.00361904, 0.0029996 , ..., 0.00832068, 0.01694544,
        0.01090639], shape=(11091,))}

Baseline даёт 0.10 — это уровень случайного ранжирования при базовой доле ботов 0.081.
Логрегрессия поднимает метрику до ~0.51, бустинг до ~0.74. Разрыв между ними означает, что
связи нелинейные и важны взаимодействия признаков.

## 8. Что даёт запрещённый признак

Прежде чем улучшать модель, зафиксируем цену отказа от утечки — чтобы решение было осознанным,
а не случайным.

In [12]:
X_leak = X_train.copy()
X_leak['n_captcha_after_window'] = cap.reindex(X_leak.cookie_id).fillna(0).values

honest = report(X_train, FEATURES, 'честно (только события в окне)', params=LGB_BASE)
leaky = report(X_leak, FEATURES + ['n_captcha_after_window'], 'С УТЕЧКОЙ (капча после окна)', params=LGB_BASE)
print(f'\nцена отказа от утечки: {leaky["pr"] - honest["pr"]:+.4f} метрики')

честно (только события в окне)         P@R0.7 = 0.7367 +- 0.0096 | ROC 0.9273 | PR-AUC 0.7736


С УТЕЧКОЙ (капча после окна)           P@R0.7 = 0.8898 +- 0.0031 | ROC 0.9455 | PR-AUC 0.8398

цена отказа от утечки: +0.1532 метрики


Утечка стоит +0.15 метрики (0.737 → 0.890). Для сравнения, все осмысленные улучшения модели
дали сотые доли. Поэтому такой признак легко получить случайно и не заметить: локальная
валидация выглядит прекрасно, а в проде признака просто нет.

## 9. Отбор признаков

72 признака на 899 положительных примеров — много, часть из них шумит. Важность считается
усреднением по фолдам (устойчивее, чем одна модель на всём train), затем проверяется несколько
размеров топа.

In [13]:
def cv_importance(X, cols, params, n_repeats=2):
    imp = pd.Series(0.0, index=cols)
    for r in range(n_repeats):
        for tr_idx, _ in StratifiedKFold(5, shuffle=True, random_state=100 + r).split(X, y):
            m = lgb.LGBMClassifier(random_state=r, n_jobs=-1, verbose=-1, **params)
            m.fit(X.iloc[tr_idx][cols], y[tr_idx])
            imp += pd.Series(m.booster_.feature_importance('gain'), index=cols)
    return (imp / imp.sum()).sort_values(ascending=False)

IMP = cv_importance(X_train, FEATURES, LGB_BASE)
print('топ-20 признаков по вкладу (% от общего gain):')
print((IMP * 100).head(20).round(2).to_string())
print('\nсамые слабые:')
print((IMP * 100).tail(8).round(3).to_string())

топ-20 признаков по вкладу (% от общего gain):
dt_median           13.48
pointer_x_std       12.28
dt_p90               4.82
dt_item_median       4.44
loc_nunique          4.21
dt_p10               3.17
sh_engagement        3.03
cat_nunique          2.90
search_page_mean     2.75
item_nunique         2.52
sh_contact           2.26
n_events             2.17
cookie_age_days      2.14
dt_long_share        1.98
top_cat_share        1.84
item_repeat_rate     1.77
top_loc_share        1.57
dt_cv                1.46
pointer_y_std        1.43
pointer_coverage     1.38

самые слабые:
n_active_hours            0.098
same_ts_share             0.071
n_contact_phone_show      0.070
n_contact_chat_open       0.021
n_login                   0.021
n_contact_message_sent    0.020
ua_headless               0.004
ua_bot_token              0.000


In [14]:
print(f'{"K признаков":>12s} {"OOF P@R0.7":>20s} {"ROC":>8s} {"time-holdout":>14s}')
print('-' * 58)
for K in [20, 30, 50, len(FEATURES)]:
    cols = list(IMP.head(K).index)
    r = evaluate(X_train, cols, params=LGB_BASE, n_repeats=2)
    th, _, _ = time_holdout(X_train, cols, LGB_BASE)
    print(f'{K:>12d} {r["pr"]:>12.4f} +- {r["pr_std"]:.4f} {r["roc"]:>8.4f} {th:>14.4f}')

 K признаков           OOF P@R0.7      ROC   time-holdout
----------------------------------------------------------


          20       0.7462 +- 0.0059   0.9294         0.7130


          30       0.7464 +- 0.0063   0.9283         0.7097


          50       0.7333 +- 0.0141   0.9270         0.7333


          72       0.7223 +- 0.0146   0.9269         0.7311


ROC-AUC почти не меняется при любом K (0.927–0.930): ранжирование насыщено, а скачки P@R — шум
на хвосте распределения. Различия между вариантами лежат в пределах 1–1.5 сигмы.

Беру K=30 не из-за максимума на валидации, а как компромисс: держится по обеим схемам и вдвое
компактнее полного набора, то есть меньше риск переобучения.

## 10. Гиперпараметры

Случайный поиск по 15 конфигурациям, отдельным прогоном (см. `README.md`).

Лучшая конфигурация на 2 повторах давала 0.7596, но на 6 повторах прирост ужался до 0.7460
против 0.7430 у базовых параметров. Почти весь выигрыш оказался отбором максимума из шумных
замеров.

| Конфигурация | OOF (6 повторов) | time-holdout |
|---|---|---|
| K=30, базовые параметры | 0.7430 ± 0.0102 | 0.7097 |
| K=30, подобранные | 0.7460 ± 0.0155 | 0.7196 |
| все 72, базовые | 0.7300 ± 0.0110 | 0.7264 |
| все 72, подобранные | 0.7297 ± 0.0082 | 0.7176 |

Беру вторую строку. Ячейка ниже пересчитывает метрику в ноутбуке и даёт 0.7438 — расхождение
из-за других сидов, внутри сигмы.

In [15]:
LGB_FINAL = dict(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=80,
                 subsample=0.7, subsample_freq=1, colsample_bytree=0.9, reg_lambda=5.0,
                 scale_pos_weight=1.0)

SELECTED = list(IMP.head(30).index)
print('финальный набор признаков (30):')
for i, c in enumerate(SELECTED, 1):
    print(f'{i:2d}. {c:26s} {IMP[c] * 100:5.2f}%')

финальный набор признаков (30):
 1. dt_median                  13.48%
 2. pointer_x_std              12.28%
 3. dt_p90                      4.82%
 4. dt_item_median              4.44%
 5. loc_nunique                 4.21%
 6. dt_p10                      3.17%
 7. sh_engagement               3.03%
 8. cat_nunique                 2.90%
 9. search_page_mean            2.75%
10. item_nunique                2.52%
11. sh_contact                  2.26%
12. n_events                    2.17%
13. cookie_age_days             2.14%
14. dt_long_share               1.98%
15. top_cat_share               1.84%
16. item_repeat_rate            1.77%
17. top_loc_share               1.57%
18. dt_cv                       1.46%
19. pointer_y_std               1.43%
20. pointer_coverage            1.38%
21. platform_nunique            1.34%
22. query_repeat_rate           1.30%
23. cat_entropy                 1.26%
24. events_per_active_hour      1.24%
25. dt_item_min                 1.24%
26. sh_seller_pro 

In [16]:
final = report(X_train, SELECTED, 'ФИНАЛ: K=30 + подобранные параметры', params=LGB_FINAL, n_repeats=6)
th, th_auc, th_pos = time_holdout(X_train, SELECTED, LGB_FINAL)
print(f'{"":38s} time-holdout P@R0.7 = {th:.4f} (ROC {th_auc:.4f}, {th_pos} ботов)')

# диагностическая метрика из metric.py: сколько ботов ловим, задев не более 1% людей
oof_final = final['oof']
print(f'{"":38s} recall@FPR=1%      = {recall_at_fpr(y, oof_final, 0.01):.4f}')

ФИНАЛ: K=30 + подобранные параметры    P@R0.7 = 0.7438 +- 0.0084 | ROC 0.9290 | PR-AUC 0.7763


                                       time-holdout P@R0.7 = 0.7196 (ROC 0.9274, 220 ботов)
                                       recall@FPR=1%      = 0.6107


## 11. Что пробовал и от чего отказался

| Гипотеза | Результат | Решение |
|---|---|---|
| Межкуковые признаки: общий User-Agent в пределах дня, популярность объявлений и запросов | OOF 0.7410 → 0.7477 при σ 0.005–0.011 | Отклонил: в пределах шума, а пайплайн усложняется |
| Ещё 10 поведенческих признаков: паузы между `item_view`, повтор запроса, переходы между типами событий | OOF 0.741 → 0.734, ROC без изменений | Часть прошла отбор по важности, остальные отсеялись |
| Подбор гиперпараметров, 15 конфигураций | 0.7460 против 0.7430 у базовых | Принял, но прирост небольшой |
| `scale_pos_weight` = 3 и 6 против дисбаланса | Без улучшения | Отклонил |
| Сильная регуляризация: `num_leaves=7`, `reg_lambda=20` | OOF 0.7316, holdout 0.6937 | Отклонил |
| ExtraTrees и смесь с LightGBM по рангам | Не лучше одного LightGBM | Отклонил |
| Признак `captcha_shown` | +0.15 метрики | Отклонил как утечку |

Самым полезным изменением за всю работу оказался переход от усреднения метрики по фолдам к
подсчёту на полном OOF-векторе. Разброс упал с ±0.09 до ±0.01, и сравнения вариантов стали
осмысленными.

## 12. Финальная модель и предсказание

Модель обучается на всём train и усредняется по 5 сидам. Все источники случайности зафиксированы,
поэтому повторный запуск даёт идентичный `submission.csv`.

In [17]:
N_SEEDS = 5
test_pred = np.zeros(len(X_test))
for s in range(N_SEEDS):
    model = lgb.LGBMClassifier(random_state=s, n_jobs=-1, verbose=-1, **LGB_FINAL)
    model.fit(X_train[SELECTED], y)
    test_pred += model.predict_proba(X_test[SELECTED])[:, 1]
test_pred /= N_SEEDS

submission = pd.DataFrame({'cookie_id': X_test.cookie_id.values, 'score': test_pred})

# --- проверки формата: ровно одна строка на каждую куку из test, без пропусков и дубликатов ---
sample = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample.columns), 'не те колонки'
assert len(submission) == len(test), 'не то число строк'
assert submission.cookie_id.nunique() == len(test), 'дубликаты cookie_id'
assert set(submission.cookie_id) == set(test.cookie_id), 'состав cookie_id не совпадает с test'
assert submission.score.notna().all(), 'есть пропуски в score'
assert submission.score.between(0, 1).all(), 'score вне [0, 1]'

submission.to_csv('submission.csv', index=False)
print('submission.csv сохранён:', submission.shape)
print(submission.score.describe().round(4).to_string())
print('\nдоля куки со score > 0.5:', round((submission.score > 0.5).mean(), 4),
      '| ожидаемая доля ботов ~', round(y.mean(), 4))
print()
print(submission.head())

submission.csv сохранён: (4909, 2)
count    4909.0000
mean        0.0823
std         0.2105
min         0.0003
25%         0.0039
50%         0.0100
75%         0.0340
max         0.9977

доля куки со score > 0.5: 0.0629 | ожидаемая доля ботов ~ 0.0811

             cookie_id     score
0  ck_315fb710a0e371e7  0.000725
1  ck_a76ee3b3e3e522fd  0.018639
2  ck_94c9a4d382689e82  0.022423
3  ck_8eaf9509ad9462a0  0.000971
4  ck_9a88a5a989cb5bc6  0.004126


In [18]:
# Распределение скоров на train и test должно быть похожим — сильное расхождение
# означало бы, что модель встретила на тесте что-то незнакомое.
print('квантили скоров:')
q = [0.5, 0.75, 0.9, 0.95, 0.99]
print(pd.DataFrame({
    'train_OOF': np.quantile(oof_final, q),
    'test': np.quantile(test_pred, q),
}, index=[f'p{int(v * 100)}' for v in q]).round(4).to_string())

квантили скоров:
     train_OOF    test
p50     0.0086  0.0100
p75     0.0292  0.0340
p90     0.1372  0.1856
p95     0.6363  0.7038
p99     0.9854  0.9875


## 13. Ограничения решения

1. Метрика нестабильна: разброс OOF по сидам ±0.01–0.015. Разницу меньше ~0.02 между вариантами
   я не считаю значимой, и результат на скрытом тесте может отличаться примерно на столько же.
2. Разметка — прокси. Положительный класс собран по известным сервисам сбора данных, поэтому
   неизвестные боты почти наверняка попали в отрицательный класс, а настоящий recall ниже
   измеренного.
3. Модель видит ровно сутки. Медленный парсер, размазанный по неделям, останется незамеченным.
4. Сильные признаки обходятся: паузы и движения курсора имитируются, если знать, что их считают.
   Нужно регулярное переобучение и признаки, которые подделать дороже.
5. Порог я не выбираю — метрика перебирает его сама, важно только ранжирование. Для прода порог
   подбирается под допустимую долю ложных срабатываний (`recall_at_fpr` в `metric.py`).